In [28]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain import hub
from langchain.tools import tool
import requests

In [29]:
from langchain.agents import create_react_agent, AgentExecutor


In [30]:
load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")

In [31]:
search_tool = TavilySearchResults(max_results=2)
llm = ChatGoogleGenerativeAI(
    model="gemini-3-flash-preview",
    temperature=0.3,
)
# result = search_tool.invoke("Give me the latest news on AI")
# result
# res=llm.invoke("what is date today")
# print("res",res.content)

In [32]:
prompt = hub.pull("hwchase17/react")



/Users/aakash-solanki/Desktop/Agentic AI/Langchain_agent/langagentNew/lib/python3.9/site-packages/langsmith/client.py:7618: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  prompt = loads(json.dumps(prompt_object.manifest))


In [33]:
@tool
def get_weather_data(city: str) -> str:
    """
    Fetch current weather information for a city.
    """

    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response = requests.get(url)

    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    return (
        data
        # f"City: {city}\n"
        # f"Temperature: {data['current']['temperature']}°C\n"
        # f"Weather: {data['current']['weather_descriptions'][0]}\n"
        # f"Humidity: {data['current']['humidity']}%"
    )

In [34]:
tools = [search_tool,get_weather_data]
# ==========================================
# CREATE AGENT
# ==========================================

agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)
# ==========================================
# EXECUTOR
# ==========================================

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True

)
# ==========================================
# RUN
# ==========================================

response = agent_executor.invoke({
    "input": (
        "Find the capital of India"
        "and then find its current weather."
    )
})



> Entering new AgentExecutor chain...
Action: tavily_search_results_json
Action Input: capital of India[{'title': 'List of capitals of India - Wikipedia', 'url': 'https://en.wikipedia.org/wiki/List_of_capitals_of_India', 'content': '## Modern period\n\n[edit]\n\n In 1858, Allahabad the capital of India for a day when it also served as the capital of North-Western Provinces.\n During the British Raj, until 1911, Calcutta was the capital of India.\n By the latter half of the 19th century, Shimla had become the summer capital.\n King George V proclaimed the transfer of the capital from Calcutta to Delhi at the climax of the 1911 Delhi Durbar on 12 December 1911. The buildings housing the Viceroy, government, and parliament were inaugurated in early 1931.\n\n## References\n\n[edit] [...] Appearance\n\nFrom Wikipedia, the free encyclopedia\n\n|  |\n\n| This article needs more citations. Please help improve this article by adding citations to reliable sources. Unsourced material may be cha

In [35]:
print(response["output"])


The capital of India is New Delhi. The current weather in New Delhi is 24°C with patchy rain nearby.
